In [0]:
import os

ano_mes = '2026_06'
volumes = os.listdir('/Volumes/cnpj_data_lakehouse/gold') #bridge, dim, fact, int
  
for volume in volumes:
  if(volume == 'int'):
    continue 
      
  files = os.listdir(f"/Volumes/cnpj_data_lakehouse/gold/{volume}/{ano_mes}")       
      
  for file in files:
    table_name = file.replace('.parquet','')
    
    spark.sql(f"DROP TABLE IF EXISTS cnpj_data_lakehouse.gold.{table_name}")
    spark.sql(f"CREATE OR REPLACE TABLE cnpj_data_lakehouse.gold.{table_name} USING DELTA AS SELECT * FROM parquet.`/Volumes/cnpj_data_lakehouse/gold/{volume}/{ano_mes}/{file}`")   

    match table_name:
      case 'bridge_empresas_socios':
        spark.sql(f"ALTER TABLE cnpj_data_lakehouse.gold.{table_name} ALTER COLUMN sk_empresa_id SET NOT NULL")
        spark.sql(f"ALTER TABLE cnpj_data_lakehouse.gold.{table_name} ALTER COLUMN sk_socio_id SET NOT NULL")
        spark.sql(f"ALTER TABLE cnpj_data_lakehouse.gold.{table_name} ALTER COLUMN sk_tipos_pessoas SET NOT NULL")
        spark.sql(f"ALTER TABLE cnpj_data_lakehouse.gold.{table_name} ADD CONSTRAINT pk_{table_name}_sk_id PRIMARY KEY (sk_empresa_id, sk_socio_id)")

      case 'bridge_estabelecimentos_cnaes':
        spark.sql(f"ALTER TABLE cnpj_data_lakehouse.gold.{table_name} ALTER COLUMN sk_estabelecimento_id SET NOT NULL")
        spark.sql(f"ALTER TABLE cnpj_data_lakehouse.gold.{table_name} ALTER COLUMN sk_cnae_id SET NOT NULL")        
        spark.sql(f"ALTER TABLE cnpj_data_lakehouse.gold.{table_name} ADD CONSTRAINT pk_{table_name}_sk_id PRIMARY KEY (sk_estabelecimento_id, sk_cnae_id)")

      case _:    
        spark.sql(f"ALTER TABLE cnpj_data_lakehouse.gold.{table_name} ALTER COLUMN sk_id SET NOT NULL")
        spark.sql(f"ALTER TABLE cnpj_data_lakehouse.gold.{table_name} ADD CONSTRAINT pk_{table_name}_sk_id PRIMARY KEY (sk_id)")

In [0]:
%sql
--bridge_empresas_socios
ALTER TABLE cnpj_data_lakehouse.gold.bridge_empresas_socios ADD CONSTRAINT fk_bridge_empresas_socios_dim_empresas FOREIGN KEY (sk_empresa_id) REFERENCES cnpj_data_lakehouse.gold.dim_empresas(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.bridge_empresas_socios ADD CONSTRAINT fk_bridge_empresas_socios_dim_socios FOREIGN KEY (sk_socio_id) REFERENCES cnpj_data_lakehouse.gold.dim_socios(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.bridge_empresas_socios ADD CONSTRAINT fk_bridge_empresas_socios_dim_tempo_entrada_sociedade FOREIGN KEY (sk_data_entrada_sociedade) REFERENCES cnpj_data_lakehouse.gold.
dim_tempo_entrada_sociedade(sk_id);

--bridge_estabelecimentos_cnaes
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_cnaes ADD CONSTRAINT fk_bridge_estabelecimentos_cnaes_dim_estabelecimento_empresas FOREIGN KEY (sk_estabelecimento_id) REFERENCES cnpj_data_lakehouse.gold.dim_estabelecimentos(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.bridge_estabelecimentos_cnaes ADD CONSTRAINT fk_bridge_estabelecimentos_cnaes_dim_cnaes FOREIGN KEY (sk_cnae_id) REFERENCES cnpj_data_lakehouse.gold.dim_cnaes(sk_id);

--fact_empresas
ALTER TABLE cnpj_data_lakehouse.gold.fact_empresas ADD CONSTRAINT fk_fact_empresas_dim_empresas FOREIGN KEY (sk_id) REFERENCES cnpj_data_lakehouse.gold.dim_empresas(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_empresas ADD CONSTRAINT fk_fact_empresas_dim_portes_empresas FOREIGN KEY (sk_portes_empresas) REFERENCES cnpj_data_lakehouse.gold.dim_portes_empresas(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_empresas ADD CONSTRAINT fk_fact_empresas_dim_is_mei FOREIGN KEY (sk_mei) REFERENCES cnpj_data_lakehouse.gold.dim_is_mei(sk_id);

--fact_estabelecimentos
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_estabelecimentos FOREIGN KEY (sk_id) REFERENCES cnpj_data_lakehouse.gold.dim_estabelecimentos(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_empresas FOREIGN KEY (sk_empresas) REFERENCES cnpj_data_lakehouse.gold.dim_empresas(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_tipos_estabelecimentos FOREIGN KEY (sk_tipos_estabelecimentos) REFERENCES cnpj_data_lakehouse.gold.dim_tipos_estabelecimentos(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_situacoes_cadastrais FOREIGN KEY (sk_situacoes_cadastrais) REFERENCES cnpj_data_lakehouse.gold.dim_situacoes_cadastrais(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_situacoes_cadastrais_motivos FOREIGN KEY (sk_situacoes_cadastrais_motivos) REFERENCES cnpj_data_lakehouse.gold.dim_situacoes_cadastrais_motivos(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_situacoes_especiais FOREIGN KEY (sk_situacoes_especiais) REFERENCES cnpj_data_lakehouse.gold.dim_situacoes_especiais(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_ufs FOREIGN KEY (sk_ufs) REFERENCES cnpj_data_lakehouse.gold.dim_ufs(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_municipios FOREIGN KEY (sk_municipios) REFERENCES cnpj_data_lakehouse.gold.dim_municipios(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_municipios_exterior FOREIGN KEY (sk_municipios_exterior) REFERENCES cnpj_data_lakehouse.gold.dim_municipios_exterior(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_paises FOREIGN KEY (sk_paises) REFERENCES cnpj_data_lakehouse.gold.dim_paises(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_tempo_inicio_atividade FOREIGN KEY (sk_data_inicio_atividade) REFERENCES cnpj_data_lakehouse.gold.dim_tempo_inicio_atividade(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_tempo_situacao_cadastral FOREIGN KEY (sk_data_situacoes_cadastrais) REFERENCES cnpj_data_lakehouse.gold.dim_tempo_situacao_cadastral(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_estabelecimentos ADD CONSTRAINT fk_fact_estabelecimentos_dim_tempo_situacao_especial FOREIGN KEY (sk_data_situacoes_especiais) REFERENCES cnpj_data_lakehouse.gold.dim_tempo_situacao_especial(sk_id);

--fact_socios
ALTER TABLE cnpj_data_lakehouse.gold.fact_socios ADD CONSTRAINT fk_fact_socios_dim_socios FOREIGN KEY (sk_id) REFERENCES cnpj_data_lakehouse.gold.dim_socios(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_socios ADD CONSTRAINT fk_fact_socios_dim_tipos_pessoas FOREIGN KEY (sk_tipo_pessoa) REFERENCES cnpj_data_lakehouse.gold.dim_tipos_pessoas(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_socios ADD CONSTRAINT fk_fact_socios_dim_qualificacoes FOREIGN KEY (sk_qualificacao_socio) REFERENCES cnpj_data_lakehouse.gold.dim_qualificacoes(sk_id);
ALTER TABLE cnpj_data_lakehouse.gold.fact_socios ADD CONSTRAINT fk_fact_socios_dim_faixas_etarias FOREIGN KEY (sk_faixa_etaria_socio) REFERENCES cnpj_data_lakehouse.gold.dim_faixas_etarias(sk_id);  
ALTER TABLE cnpj_data_lakehouse.gold.fact_socios ADD CONSTRAINT fk_fact_socios_dim_paises FOREIGN KEY (sk_pais_socio) REFERENCES cnpj_data_lakehouse.gold.dim_paises(sk_id); 
ALTER TABLE cnpj_data_lakehouse.gold.fact_socios ADD CONSTRAINT fk_fact_socios_dim_qualificacoes_representante FOREIGN KEY (sk_qualificacao_representante) REFERENCES cnpj_data_lakehouse.gold.dim_qualificacoes(sk_id);